# 03 — Statistical Analysis

## Goal
Formal hypothesis tests with effect sizes — statistics, not just ML.

In [ ]:
from pathlib import Path
import sys

# Locate the project root (works whether the notebook is run from notebooks/ or root)
root = Path.cwd()
while not (root / "src" / "data_loader.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("project root:", root)


In [ ]:
import pandas as pd
from scipy import stats
from src.data_loader import load_clean, TARGET
from src.statistical_analysis import cramers_v, cohens_d

df = load_clean()
table = pd.crosstab(df["PAY_0"], df[TARGET])
chi2, p, dof, _ = stats.chi2_contingency(table)
print(f"H1 repayment status vs default: chi2={chi2:.1f}, p={p:.2e}, Cramér's V={cramers_v(table):.3f}")


In [ ]:
a, b = df.loc[df[TARGET] == 1, "LIMIT_BAL"], df.loc[df[TARGET] == 0, "LIMIT_BAL"]
t, p_t = stats.ttest_ind(a, b, equal_var=False)
u, p_u = stats.mannwhitneyu(a, b)
print(f"H2 LIMIT_BAL: Welch t={t:.2f} (p={p_t:.2e}) | MWU p={p_u:.2e} | Cohen's d={cohens_d(a, b):.3f}")

a, b = df.loc[df[TARGET] == 1, "AGE"], df.loc[df[TARGET] == 0, "AGE"]
t, p_t = stats.ttest_ind(a, b, equal_var=False)
print(f"H3 AGE       : Welch t={t:.2f} (p={p_t:.2e}) | Cohen's d={cohens_d(a, b):.3f}")


In [ ]:
results = pd.read_csv(root / "reports" / "statistical_tests.csv")
results[["hypothesis", "p_value", "effect_size", "effect_type"]]


**Reading the results**
- Repayment status (esp. PAY_0) shows a *large* association with default (Cramér's V = 0.42), decaying with recency.
- Credit limit differs meaningfully (d = -0.38).
- Age is statistically significant (p = 0.023) but the effect is *negligible* (d = 0.03) — a textbook large-n lesson: p-values are not effect sizes.